# Status Report 2024/10/23
## Progress
1. Basic Understanding of TLD IoT Hamburg / OGC SensorThingsAPI
2. Crude Implementation of a DataFetcher class

## Problems
1. Connections Times
2. observedArea vs. Location
3. General Lack of traffic light data

## Goals
1. Implement method for finding ID by coords
2. Implement local processing of data 
3. Implement methods for handling Ingress/Egress lanes

## Resources
1. https://docs.ogc.org/is/18-088/18-088.html (OGC SensorThingsAPI Dokumentation)
2. TLD IoT Hamburg Usage Guide

## Licenses
1. OGC SensorThingsAPI is License Free

In [4]:
from DataFetcher import DataFetcher
import json

In [ ]:
data_fetcher = DataFetcher()

endpoint = "Datastreams"
params = {
    "$filter": "properties/layerName eq 'primary_signal'",
    "$expand" : "Thing($expand=Locations)",
}

data_fetcher.fetch_data(endpoint,params)

print(f"Retrieved {len(data_fetcher.get_data())} datastreams.")

In [ ]:
print(json.dumps(data_fetcher.get_data()[4],indent=4))

In [ ]:
import folium
from IPython.display import display

def switch_coords(coords):
    temp = coords[0]
    coords[0] = coords[1]
    coords[1] = temp

#save possible colors for markers
colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black']
def create_marker(coords,color_index):
    folium.CircleMarker(location=coords,radius=1,color = colors[color_index], fill = True, fill_color = colors[color_index], opacity = 1).add_to(map_object)
#define starting point for map (chosen arbitrarily)
hamburg_mid_point = [53.571974, 9.969119]

#create maps
map_object = folium.Map(location=hamburg_mid_point, zoom_start=12)


counter = 0
for stream in data_fetcher.get_data():
    if "observedArea" not in stream.keys():
        continue
    
    if stream["observedArea"]["type"] == "Point":
        coords = stream["observedArea"]["coordinates"]
        switch_coords(coords)
        create_marker(coords,0)
    elif stream["observedArea"]["type"] == "LineString":
        coord_pairs = stream["observedArea"]["coordinates"]
        for coords in coord_pairs:
            switch_coords(coords)  # latidute and longitude are given switch up by the API
            #add Marker for each coordinate_pair
            create_marker(coords,1)
    else:
        coord_pairs = stream["observedArea"]["coordinates"][0]
        for coords in coord_pairs:
            switch_coords(coords)  # latidute and longitude are given switch up by the API
            create_marker(coords,2)

# Display the map
display(map_object)

In [ ]:
coords_observedArea = [[53.5805445,9.7772179],[53.5805446,9.777218],[53.5811757,9.7800438],[53.5805445,9.7772179]]
coords_Locations = [[[53.5805445, 9.7772179], [53.5805535, 9.7772171]], [[53.5805446, 9.7772173], [53.5804716, 9.7772229]], [[53.5804715, 9.7772235], [53.5804625, 9.7772243]]]


# Create a map object centered at a specific latitude and longitude
map_object = folium.Map(location=coords_observedArea[0], zoom_start=20)

colors = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'lightred', 'beige', 'darkblue', 'darkgreen', 'cadetblue', 'darkpurple', 'white', 'pink', 'lightblue', 'lightgreen', 'gray', 'black']
color_counter = 0
# Add markers to the map
for coord in coords_observedArea:
    folium.CircleMarker(location=coord,radius=5,color = colors[color_counter], fill = True, fill_color = colors[color_counter], opacity = 0.5).add_to(map_object)

color_counter += 1

for LineString in coords_Locations:
    for coord in LineString:
        folium.CircleMarker(location=coord,radius=5,color = colors[color_counter], fill = True, fill_color = colors[color_counter],opacity = 0.5).add_to(map_object)
    color_counter += 1


# Display the map
display(map_object)